# 2.9 Fourth order equations

We consider the Kirchhoff plate equation: Find $w \in H^2$, such that
$$
\int \nabla^2 w : \nabla^2 v = \int f v
$$

A conforming method requires $C^1$-continuous finite elements, which are complicated to construct. We consider the following two alternatives, which get along with simpler finite element spaces:

## Hybridized $C^0$-continuous interior penalty method:

A simple way out is to use continuous elements, and treat the missing $C^1$-continuity by a Discontinuous Galerkin method. A DG formulation is

$$
\sum_T \nabla^2 w : \nabla^2 v
- \int_{E} \{\nabla^2 w\}_{nn} \, [\partial_n v]
- \int_{E} \{\nabla^2 v\}_{nn} \, [\partial_n w] + \alpha \int_E  [\partial_n w]  [\partial_n v]
$$

[Baker 77, Brenner Gudi Sung, 2010]

We consider its hybrid DG version, where the normal derivative is a new, facet-based variable:

$$
\sum_T \nabla^2 w : \nabla^2 v
- \int_{\partial T} (\nabla^2 w)_{nn} \, (\partial_n v - \widehat{v_n})
- \int_{\partial T} (\nabla^2 v)_{nn} \, (\partial_n w - \widehat{w_n}) + \alpha \int_E (\partial_n v - \widehat{v_n}) (\partial_n w - \widehat{w_n})
$$

The facet variable is the normal derivative $n_E \cdot \nabla w$, which is oriented along the arbitrarily chosen edge normal vector. We cannot use the `FacetFESpace` since it does not have the orientation. We use the `NormalFacetFESpace`, which provides vector-valued functions in normal direction on the facets, like the normal traces of an $H(div)$ space.

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
mesh = Mesh (unit_square.GenerateMesh(maxh=0.1))

In [ ]:
order = 3

V1 = H1(mesh, order=order)
V2 = NormalFacetFESpace(mesh, order=order-1)
V = V1*V2

(w,what), (v,vhat) = V.TnT()

Some proxy functions and `GridFunction`s provide additional differential operators. We can get them via the `Operator` function. `w.Operator("hesse")` provides the Hessian, a matrix-valued function. Note that we can use `InnerProduct(.,.)` for $\nabla^2 w : \nabla^2 v$:

In [ ]:
n = specialcf.normal(mesh.dim)
h = specialcf.mesh_size

def jumpdn(v,vhat):
    return n*(grad(v)-vhat)
def hesse(v):
    return v.Operator("hesse")
def hessenn(v):
    return InnerProduct(n, hesse(v)*n)

dS = dx(element_boundary=True)
lhs = InnerProduct (hesse(w), hesse(v)) * dx \
     - hessenn(w) * jumpdn(v,vhat) * dS \
     - hessenn(v) * jumpdn(w,what) * dS \
     + 3*order*order/h * jumpdn(w,what) * jumpdn(v,vhat) * dS

We solve the clamped plate. The boundary conditions $w = 0$ and $\partial_n w = 0$ are essential boundary conditions for the two components:

In [ ]:
gf = Solve(lhs == 1*v*dx, w[BND(".*")] == 0, what[BND(".*")] == 0)
gfw = gf[w]

Draw (gfw, mesh, "disp_DG")
Draw (grad (gfw), mesh, "grad")
Draw (hesse (gfw), mesh, "hesse");

## The Hellan-Herrmann-Johnson Mixed Method:
[Hellan 67, Herrmann 67, Johnson 73, Arnold+Brezzi 85, Comodi 89]

We introduce the bending moments as a new, matrix-valued variable $\sigma = \nabla^2 w$. Then we can write the plate equation as a saddle point problem:

$$
\DeclareMathOperator{\Div}{div}
\begin{array}{ccccll}
\int \sigma : \tau & - & \int \tau : \nabla^2 w & = & 0 & \forall \, \tau \\
-\int \sigma : \nabla^2 v &  &  & = & -\int f v  & \forall \, v
\end{array}
$$

Integration by parts in the upper-right and lower-left terms:

$$
\DeclareMathOperator{\Div}{div}
\begin{array}{ccccll}
\int \sigma : \tau & + & < \Div \tau , \nabla w > & = & 0 & \forall \, \tau \\
< \Div \sigma , \nabla v >  &  &  & = & -\int f v  & \forall \, v
\end{array}
$$

The Hellan-Herrmann-Johnson method uses finite elements $\sigma_h$ which have a continuous normal-normal component. Then, the term $<\Div \sigma, \nabla v>$ must be understood in distributional form:

$$
\sum_T \int_T \Div \sigma \, \nabla v - \sum_E \int_E [\sigma_{nt}] (\nabla v)_t
$$

Since $\sigma_{nn}$ is continuous, the jump occurs only in the tangential component of $\sigma_n$. Luckily, it hits the tangential component of $\nabla v$, which is single-valued for $H^1$ finite elements. Thus, we can rewrite the term element by element:

$$
\sum_T \int_T \Div \sigma \, \nabla v -  \int_{\partial T} \sigma_{nt}\nabla_t v
$$

The Hellan-Herrmann-Johnson method does not require access to the neighbour element as in DG methods.
In [Pechstein-Schöberl 11] the function space $H(\Div \Div)$ was introduced. The HHJ finite elements are (nearly) conforming for this space. A basis was also given in [PS11].

For the clamped plate, $w = 0$ is an essential boundary condition for the displacement. The condition $\partial_n w = 0$ is a natural boundary condition now.

In [ ]:
order = 3

V = HDivDiv(mesh, order=order-1)
Q = H1(mesh, order=order)
X = V*Q

print ("ndof-V:", V.ndof, ", ndof-Q:", Q.ndof)

(sigma, w), (tau, v) = X.TnT()

n = specialcf.normal(mesh.dim)

def tang(u): return u-(u*n)*n

lhs = (InnerProduct (sigma, tau) + div(sigma)*grad(v) + div(tau)*grad(w))*dx \
      + (-(sigma*n) * tang(grad(v)) - (tau*n)*tang(grad(w)))*dx(element_boundary=True)

gfu = Solve(lhs == -1*v*dx, w[BND(".*")] == 0)

Draw (gfu[sigma], mesh, name="sigma")
Draw (gfu[w], mesh, name="disp");

Both methods give nearly the same deflection in the center of the plate:

In [ ]:
mip = mesh(0.5, 0.5)
print ("w, C0-IP:", gfw(mip))
print ("w, HHJ:  ", gfu[w](mip))